# Latihan Soal — Notebook 03 & 04

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [1]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

ModuleNotFoundError: No module named 'sortedcontainers'

---
# Bagian 03 — Propositional Logic

Latihan soal berikut disalin dari **Notebook 03 - Propositional Logic**.


## Soal 1: Diskusi Kelompok

**Slide 27**

> Diketahui proposisi atomic $x_1$, $x_2$, $x_3$ di mana $x_i$ merupakan notasi
> untuk kalimat "pegawai i sedang bekerja". Buatlah proposisi sesuai kondisi
> berikut:
>
> 1. Salah satu dari pegawai 1, 2 dan 3 sedang bekerja.
> 2. Terdapat dua pegawai dari pegawai 1, 2, dan 3 yang sedang bekerja.
> 3. Tidak semua pegawai sedang bekerja.
>
> Jika sudah, diskusikan jawaban kelompokmu dengan kelompok lain.

Terjemahkan tiap kondisi jadi proposisi, lalu cek jawabanmu dengan fungsi
`satisfying_models` di bawah, yang mengembalikan semua model (dari 8 model
yang mungkin) yang membuat proposisimu bernilai benar. Simbolnya ditulis
`X1`, `X2`, `X3` (uppercase) di kode, mewakili $x_1$, $x_2$, $x_3$ di soal.
`pl_true` cuma mengenali simbol yang diawali huruf besar sebagai proposition
symbol.

In [2]:
def satisfying_models(sentence, syms):
    """Enumerate every model over `syms` and return the ones where `sentence` is true."""
    models = []
    for values in itertools.product([False, True], repeat=len(syms)):
        model = dict(zip(syms, values))
        if pl_true(sentence, model):
            models.append(model)
    return models


X1, X2, X3 = expr('X1, X2, X3')
print("Total kemungkinan model:", 2 ** 3)

NameError: name 'expr' is not defined

In [ ]:
syms = [X1, X2, X3]
def show(label, sentence):
    models = satisfying_models(sentence, syms)
    print(f"{label}\n  proposisi : {sentence}\n  jumlah model: {len(models)}")
    for m in models:
        print("   ", {str(k): v for k, v in m.items()})
    print()


#  Nomor 1: "Salah satu dari pegawai 1, 2, dan 3 sedang bekerja" 
# Versi A: paling sedikit satu pegawai bekerja
p1_atleast = X1 | X2 | X3
# Versi B: tepat satu pegawai bekerja
p1_exact = (X1 & ~X2 & ~X3) | (~X1 & X2 & ~X3) | (~X1 & ~X2 & X3)

show("1A. Paling sedikit satu bekerja", p1_atleast)
show("1B. Tepat satu bekerja", p1_exact)

# Nomor 2: "Terdapat dua pegawai yang sedang bekerja" 
# Versi A: paling sedikit dua pegawai bekerja
p2_atleast = (X1 & X2) | (X1 & X3) | (X2 & X3)
# Versi B: tepat dua pegawai bekerja
p2_exact = (X1 & X2 & ~X3) | (X1 & ~X2 & X3) | (~X1 & X2 & X3)

show("2A. Paling sedikit dua bekerja", p2_atleast)
show("2B. Tepat dua bekerja", p2_exact)

# Nomor 3: "Tidak semua pegawai sedang bekerja"
p3 = ~(X1 & X2 & X3)            # ekuivalen (De Morgan): ~X1 | ~X2 | ~X3
p3_wrong = ~X1 & ~X2 & ~X3      # "tidak ada yang bekerja" bukan terjemahan yang benar

show("3.  Tidak semua bekerja", p3)
show("3x. Tidak ada yang bekerja (pembanding)", p3_wrong)

<details>
<summary>Klik untuk melihat hint</summary>

Nomor 1 dan 2 sengaja ambigu -- itu bagian dari pelajarannya, jangan
diselesaikan sepihak. Coba tulis **dua** versi proposisi untuk masing-masing
("paling sedikit satu/dua" lawan "tepat satu/dua"), lalu bandingkan
`len(satisfying_models(...))` dari kedua versi itu -- beda tidak?

Nomor 3, hati-hati: "tidak semua bekerja" tidak sama dengan "tidak ada yang
bekerja". Tulis dulu $\neg(x_1 \land x_2 \land x_3)$ apa adanya (jangan buru-
buru disederhanakan), lalu bandingkan jumlah model yang memenuhinya dengan
jumlah model untuk $\neg x_1 \land \neg x_2 \land \neg x_3$.

</details>

## Soal 2

Buktikan hukum De Morgan $\neg(P \land Q) \equiv \neg P \lor \neg Q$ dengan
truth table yang dibangkitkan pakai `pl_true` untuk semua kombinasi $P$ dan
$Q$, bukan dihitung manual.

In [ ]:
P, Q = expr('P, Q')
lhs = ~(P & Q)
rhs = ~P | ~Q

rows = []
for p_val, q_val in itertools.product([False, True], repeat=2):
    model = {P: p_val, Q: q_val}
    rows.append({
        "P": p_val,
        "Q": q_val,
        "¬(P∧Q)": pl_true(lhs, model),
        "¬P∨¬Q": pl_true(rhs, model),
        "sama?": pl_true(lhs, model) == pl_true(rhs, model),
    })

demorgan_table = pd.DataFrame(rows)
print(demorgan_table)

biconditional = lhs |'<=>'| rhs
is_tautology = all(
    pl_true(biconditional, {P: p_val, Q: q_val})
    for p_val, q_val in itertools.product([False, True], repeat=2)
)
print("\n¬(P∧Q) <=> (¬P∨¬Q) tautology di semua baris?", is_tautology)

<details>
<summary>Klik untuk melihat hint</summary>

Bangun `lhs` dan `rhs` sebagai `Expr` sesuai kedua sisi persamaannya, lalu
loop `itertools.product` untuk $P$, $Q$ persis pola yang sama dengan tabel
di 3.4 -- bandingkan `pl_true` keduanya di tiap baris, jangan dihitung
manual dulu. Kalau di semua baris nilainya sama, apa artinya itu buat
kalimat $\text{lhs} \Leftrightarrow \text{rhs}$ (lihat "Vocabulary
tambahan" di 3.4)?

</details>

## Soal 3

Di 3.2 disebutkan implikasi ($\Rightarrow$) adalah konektif yang jadi bentuk
*rule* KB nanti di Notebook 04, misalnya kalimat seperti "kalau ada breeze di
suatu kotak, maka ada pit di salah satu kotak tetangganya." Implikasi
material tidak mensyaratkan premise dan conclusion saling berhubungan makna
(3.4): implikasi otomatis bernilai benar begitu premise-nya salah, berapa
pun nilai conclusion-nya.

Jelaskan dengan kata-katamu sendiri: kenapa sifat ini justru yang membuat
implikasi material cocok dipakai sebagai bentuk rule KB Wumpus World, bukan
gangguan yang perlu dihindari? Bayangkan agent memakai rule "kalau breeze di
$[x,y]$, maka pit di salah satu tetangganya" — apa yang salah dengan KB
seorang agent kalau rule itu justru dianggap **salah** (bukan otomatis
benar) setiap kali agent tidak merasakan breeze di $[x,y]$?

Jawaban tidak cukup satu baris: kaitkan dengan tabel implikasi di 3.4, dan
pakai alasan yang spesifik ke skenario Wumpus, bukan cuma definisi umum
implikasi material.

In [ ]:
'''
Menurut tabel kebenaran implikasi di 3.4, P --> Q hanya salah dalam satu keadaan: $P$ benar tapi $Q$ salah.

| P (ada breeze di [x,y]) | Q (ada pit di salah satu tetangga) | P --> Q |
|---|---|---|
| F | F | T |
| F | T | T |
| T | F | F |
| T | T | T |

Satu-satunya baris yang salah adalah "ada breeze, tapi tidak ada pit di sekitarnya". 
Di Wumpus World, keadaan itu memang tidak mungkin terjadi. 
Jadi rule ini hanya bertugas melarang keadaan tersebut. 
Rule ini tidak mengatakan apa-apa tentang kotak yang tidak ada breeze-nya.
Rule ini adalah hukum dunia yang berlaku untuk semua kotak, bukan laporan tentang apa yang sedang dirasakan agent. 
Padahal kebanyakan kotak tidak punya breeze, misalnya [1,1] di awal permainan. 
Kalau rule dianggap salah setiap kali tidak ada breeze, rule itu akan salah di hampir semua kotak. 
Hukum yang tidak berlaku di kebanyakan tempat tentu bukan hukum.
Jika rule daianggap salah maka KB jadi kontradiktif KB hanya benar kalau semua kalimat di dalamnya benar. 
Misalkan agent di [1,1] tidak merasakan breeze ~B_{1,1}. Kalau rule B_{1,1} --> (P_{1,2} V P_{2,1})$ dianggap salah, KB tidak mungkin benar di dunia mana pun. 
KB yang kontradiktif bisa "membuktikan" apa saja, bahkan P_{1,2} dan ~P_{1,2} sekaligus. Akibatnya, agent tidak bisa lagi membedakan kotak aman dan kotak berbahaya. 
Hal ini juga akan bertentangan dengan tabel. Rule hanya salah kalau ada breeze tapi tidak ada pit. 
Kalau rule dianggap salah saat tidak ada breeze, itu sama saja dengan bilang "tidak ada breeze" berarti "ada breeze tapi tidak ada pit". 
Kalau tidak ada breeze, rule ini memang tidak memberi informasi apa pun, dan itu wajar. 
Kesimpulan "tidak ada breeze --> tetangga aman" datang dari arah sebaliknya, yaitu "kalau ada pit di tetangga --> pasti ada breeze".
Karena itu, di KB Wumpus aturannya biasanya ditulis dua arah (biimplikasi):

B_{1,1} <==> (P_{1,2} V P_{2,1})

Masing-masing arah tetap benar saat premise-nya salah. Tapi dengan menggabungkan keduanya, agent bisa menyimpulkan bahwa [1,2] dan [2,1] aman.
Kesimpulan: implikasi material cocok untuk rule KB karena ia hanya membuang keadaan yang benar-benar melanggar aturan dunia, dan membiarkan keadaan lain tetap mungkin.
'''

'\nMenurut tabel kebenaran implikasi di 3.4, P --> Q hanya salah dalam satu keadaan: $P$ benar tapi $Q$ salah.\n\n| P (ada breeze di [x,y]) | Q (ada pit di salah satu tetangga) | P --> Q |\n|---|---|---|\n| F | F | T |\n| F | T | T |\n| T | F | F |\n| T | T | T |\n\nSatu-satunya baris yang salah adalah "ada breeze, tapi tidak ada pit di sekitarnya". \nDi Wumpus World, keadaan itu memang tidak mungkin terjadi. \nJadi rule ini hanya bertugas melarang keadaan tersebut. \nRule ini tidak mengatakan apa-apa tentang kotak yang tidak ada breeze-nya.\nRule ini adalah hukum dunia yang berlaku untuk semua kotak, bukan laporan tentang apa yang sedang dirasakan agent. \nPadahal kebanyakan kotak tidak punya breeze, misalnya [1,1] di awal permainan. \nKalau rule dianggap salah setiap kali tidak ada breeze, rule itu akan salah di hampir semua kotak. \nHukum yang tidak berlaku di kebanyakan tempat tentu bukan hukum.\nJika rule daianggap salah maka KB jadi kontradiktif KB hanya benar kalau semua kalimat

<details>
<summary>Klik untuk melihat hint</summary>

Bayangkan implikasi didefinisikan ulang supaya bernilai **false** ketika
premise-nya false (kebalikan dari sekarang). KB adalah conjunction dari
semua rule yang di-tell ke dalamnya (Notebook 02). Kalau ada **satu saja**
kotak tanpa breeze, apa yang terjadi pada rule breeze-pit untuk kotak itu
di bawah definisi baru ini -- dan lewat operator AND, apa yang terjadi ke
**seluruh** KB?

Cek juga definisi entailment dari Notebook 02, $M(KB) \subseteq
M(\alpha)$: apa yang terjadi pada definisi itu kalau $M(KB)$ ternyata
kosong di setiap model?

</details>

---
# Bagian 04 — Forward & Backward Chaining

Latihan soal berikut disalin dari **Notebook 04 - Forward & Backward
Chaining**. Sel di bawah ini terlebih dahulu mendefinisikan fungsi
`make_kb`, `ask_forward`, `ask_backward`, `visualize_forward`, dan
`visualize_backward` yang dipakai pada soal-soal berikut (sama dengan
sel "Fungsi yang digunakan" pada Notebook 04).


---
## Fungsi yang digunakan

Jalankan sel berikut **sekali saja** setelah Setup. Sel ini menyediakan fungsi
praktikum berikut:

```python
kb = make_kb(facts, rules)
hasil_fc, trace_fc = ask_forward(kb, query)
hasil_bc, trace_bc = ask_backward(kb, query)
visualize_forward(trace_fc)
visualize_backward(trace_bc)
```

Istilah yang dipakai pada pemanggilan fungsi:

- **fact** adalah sentence yang sudah diketahui benar, misalnya `"A"`;
- **rule** adalah sentence berbentuk implikasi, misalnya `"A ==> B"`;
- **query** adalah proposition yang ingin dibuktikan, misalnya `"B"`;
- **trace** adalah catatan urutan langkah yang dilakukan algoritma.

`facts` dan `rules` ditulis sebagai list string. `query` juga dapat ditulis
sebagai string sederhana. `ask_forward` dan `ask_backward` sama-sama
mengembalikan pasangan `(hasil, trace)`.

Nama fungsi di atas adalah **helper function** (fungsi bantu) untuk praktikum
ini. Algoritma di dalamnya tetap mengikuti konsep PL-FC-ENTAILS dan backward
chaining. Mahasiswa tidak perlu menyalin atau mengubah isi fungsi pada setiap
bagian. Setelah sel dijalankan, cukup buat KB dan panggil fungsi yang diperlukan.


In [ ]:
import textwrap

import matplotlib.pyplot as plt


def make_kb(facts, rules):
    """Build a PropDefiniteKB from fact and rule strings."""
    kb = PropDefiniteKB()
    for sentence in [*facts, *rules]:
        kb.tell(expr(sentence))
    return kb


def _as_expr(value):
    """Accept either an Expr or a string containing one expression."""
    return expr(value) if isinstance(value, str) else value


def _agenda_text(agenda):
    """Format an agenda with the next item to process on the right."""
    return " → ".join(str(item) for item in agenda) or "(kosong)"


def ask_forward(kb, query):
    """Run forward chaining and return (result, trace_dataframe)."""
    goal = _as_expr(query)
    count = {}
    premise_index = {}

    for clause in kb.clauses:
        if clause.op != "==>":
            continue
        premises = conjuncts(clause.args[0])
        count[clause] = len(premises)
        for premise in premises:
            premise_index.setdefault(premise, []).append(clause)

    initial_facts = [
        clause for clause in kb.clauses if is_prop_symbol(clause.op)
    ]
    agenda = list(dict.fromkeys(initial_facts))
    known = set(agenda)
    inferred = set()
    rows = []

    while agenda:
        agenda_before = list(agenda)
        fact = agenda.pop()
        step = len(rows) + 1

        if fact == goal:
            rows.append({
                "langkah": step,
                "agenda sebelum": _agenda_text(agenda_before),
                "fact diproses": str(fact),
                "rule aktif": "-",
                "conclusion baru": "-",
                "agenda sesudah": _agenda_text(agenda),
                "keputusan": "Query ditemukan; proses berhenti.",
            })
            trace = pd.DataFrame(rows)
            trace.attrs["query"] = str(goal)
            return True, trace

        if fact in inferred:
            continue

        inferred.add(fact)
        fired_rules = []
        new_conclusions = []
        known_conclusions = []

        for clause in premise_index.get(fact, []):
            count[clause] -= 1
            if count[clause] != 0:
                continue

            conclusion = clause.args[1]
            fired_rules.append(str(clause))
            if conclusion in known:
                known_conclusions.append(str(conclusion))
            else:
                known.add(conclusion)
                agenda.append(conclusion)
                new_conclusions.append(str(conclusion))

        if not fired_rules:
            decision = "Belum ada rule baru yang seluruh premise-nya terpenuhi."
        else:
            messages = []
            if new_conclusions:
                messages.append(
                    "Tambahkan " + ", ".join(new_conclusions) + " ke agenda."
                )
            if known_conclusions:
                messages.append(
                    ", ".join(known_conclusions) + " sudah diketahui; tidak ditambahkan ulang."
                )
            decision = " ".join(messages)

        rows.append({
            "langkah": step,
            "agenda sebelum": _agenda_text(agenda_before),
            "fact diproses": str(fact),
            "rule aktif": ", ".join(fired_rules) or "-",
            "conclusion baru": ", ".join(new_conclusions) or "-",
            "agenda sesudah": _agenda_text(agenda),
            "keputusan": decision,
        })

    trace = pd.DataFrame(rows)
    trace.attrs["query"] = str(goal)
    return False, trace


def visualize_forward(trace):
    """Visualize each forward-chaining decision as one horizontal step."""
    if trace.empty:
        print("Trace kosong: tidak ada fact yang dapat diproses.")
        return

    query = trace.attrs.get("query", "?")
    figure_height = max(2.4, 2.25 * len(trace))
    figure, axes = plt.subplots(
        len(trace), 1, figsize=(14, figure_height), squeeze=False
    )

    for axis, (_, row) in zip(axes[:, 0], trace.iterrows()):
        axis.set_xlim(0, 1)
        axis.set_ylim(0, 1)
        axis.axis("off")

        query_found = "Query ditemukan" in row["keputusan"]
        decision_color = "#d1fae5" if query_found else "#e0f2fe"

        axis.text(
            0.01, 0.72, f"Langkah {row['langkah']}",
            fontsize=12, fontweight="bold", va="center"
        )
        axis.text(
            0.14, 0.72,
            "Agenda sebelum\n" + textwrap.fill(row["agenda sebelum"], 22),
            fontsize=10, va="center", ha="center",
            bbox={"boxstyle": "round,pad=0.5", "facecolor": "#f3f4f6", "edgecolor": "#6b7280"},
        )
        axis.annotate("", xy=(0.38, 0.72), xytext=(0.31, 0.72),
                      arrowprops={"arrowstyle": "->", "color": "#374151", "lw": 1.5})
        axis.text(
            0.43, 0.72, f"Proses fact\n{row['fact diproses']}",
            fontsize=10, va="center", ha="center",
            bbox={"boxstyle": "round,pad=0.5", "facecolor": "#fef3c7", "edgecolor": "#d97706"},
        )
        axis.annotate("", xy=(0.59, 0.72), xytext=(0.50, 0.72),
                      arrowprops={"arrowstyle": "->", "color": "#374151", "lw": 1.5})

        decision_text = (
            "Rule aktif: " + textwrap.fill(row["rule aktif"], 42)
            + "\nConclusion baru: " + row["conclusion baru"]
            + "\n" + textwrap.fill(row["keputusan"], 52)
        )
        axis.text(
            0.78, 0.72, decision_text,
            fontsize=9.5, va="center", ha="center",
            bbox={"boxstyle": "round,pad=0.55", "facecolor": decision_color, "edgecolor": "#0284c7"},
        )
        axis.text(
            0.14, 0.16,
            "Agenda sesudah: " + textwrap.fill(row["agenda sesudah"], 75),
            fontsize=9.5, va="center", ha="left",
        )
        axis.plot([0.01, 0.99], [0.02, 0.02], color="#d1d5db", linewidth=1)

    figure.suptitle(
        f"Visualisasi Forward Chaining — Query: {query}",
        fontsize=15, fontweight="bold", y=1.005,
    )
    figure.text(
        0.5, 0.002,
        "Agenda dibaca dari kiri ke kanan; item paling kanan diproses berikutnya.",
        ha="center", fontsize=9, color="#4b5563",
    )
    plt.tight_layout()
    plt.show()


def ask_backward(kb, query):
    """Run backward chaining and return (result, trace_dataframe)."""
    goal = _as_expr(query)
    facts = {clause for clause in kb.clauses if is_prop_symbol(clause.op)}
    rules_by_conclusion = {}

    for clause in kb.clauses:
        if clause.op == "==>":
            premises, conclusion = parse_definite_clause(clause)
            rules_by_conclusion.setdefault(conclusion, []).append(
                (clause, premises)
            )

    rows = []

    def prove(current_goal, goals_in_branch, depth, parent_step):
        step = len(rows) + 1
        row_index = len(rows)
        rows.append({
            "langkah": step,
            "parent": parent_step if parent_step is not None else "-",
            "depth": depth,
            "goal": str(current_goal),
            "rule diperiksa": "-",
            "rule berhasil": "-",
            "keputusan": "Periksa goal.",
            "hasil": None,
        })

        if current_goal in facts:
            rows[row_index]["keputusan"] = "Goal sudah tersedia sebagai fact."
            rows[row_index]["hasil"] = True
            return True

        if current_goal in goals_in_branch:
            rows[row_index]["keputusan"] = (
                "Goal yang sama sudah diperiksa pada branch ini; branch dihentikan."
            )
            rows[row_index]["hasil"] = False
            return False

        supporting_rules = rules_by_conclusion.get(current_goal, [])
        if not supporting_rules:
            rows[row_index]["keputusan"] = (
                "Tidak ada fact atau rule yang menghasilkan goal."
            )
            rows[row_index]["hasil"] = False
            return False

        next_branch = goals_in_branch | {current_goal}
        tried_rules = []

        for clause, premises in supporting_rules:
            tried_rules.append(str(clause))
            premise_results = []
            for premise in premises:
                premise_result = prove(premise, next_branch, depth + 1, step)
                premise_results.append(premise_result)
                if not premise_result:
                    break

            if all(premise_results):
                rows[row_index]["rule diperiksa"] = " | ".join(tried_rules)
                rows[row_index]["rule berhasil"] = str(clause)
                rows[row_index]["keputusan"] = (
                    "Semua premise pada rule berhasil dibuktikan."
                )
                rows[row_index]["hasil"] = True
                return True

        rows[row_index]["rule diperiksa"] = " | ".join(tried_rules)
        rows[row_index]["keputusan"] = "Semua rule pendukung gagal."
        rows[row_index]["hasil"] = False
        return False

    found = prove(goal, set(), depth=0, parent_step=None)
    trace = pd.DataFrame(rows)
    trace.attrs["query"] = str(goal)
    return found, trace


def visualize_backward(trace):
    """Visualize a backward-chaining trace as a proof-search tree."""
    if trace.empty:
        print("Trace kosong: tidak ada goal yang diperiksa.")
        return

    query = trace.attrs.get("query", "?")
    max_depth = int(trace["depth"].max())
    node_count = len(trace)
    x_spacing = 2.7
    positions = {
        int(row["langkah"]): (float(row["depth"]) * x_spacing, -int(row["langkah"]))
        for _, row in trace.iterrows()
    }

    figure_width = max(12, 5 + (max_depth + 1) * 2.35)
    figure_height = max(5, 1.25 * node_count)
    figure, axis = plt.subplots(figsize=(figure_width, figure_height))

    for _, row in trace.iterrows():
        step = int(row["langkah"])
        parent = row["parent"]
        if parent == "-":
            continue
        parent_step = int(parent)
        parent_x, parent_y = positions[parent_step]
        child_x, child_y = positions[step]
        axis.annotate(
            "",
            xy=(child_x, child_y + 0.28),
            xytext=(parent_x, parent_y - 0.28),
            arrowprops={"arrowstyle": "->", "color": "#64748b", "lw": 1.4},
        )

    for _, row in trace.iterrows():
        step = int(row["langkah"])
        x, y = positions[step]
        decision = str(row["keputusan"])
        result = bool(row["hasil"])

        if result:
            facecolor, edgecolor = "#d1fae5", "#059669"
        else:
            facecolor, edgecolor = "#fee2e2", "#dc2626"

        successful_rule = str(row["rule berhasil"])
        rule_line = (
            "\nRule: " + textwrap.fill(successful_rule, 34)
            if successful_rule != "-"
            else ""
        )
        label = (
            f"Langkah {step} | Goal: {row['goal']}"
            f"\nHasil: {result}"
            f"{rule_line}"
            f"\n{textwrap.fill(decision, 38)}"
        )
        axis.text(
            x, y, label,
            ha="center", va="center", fontsize=9.2,
            bbox={
                "boxstyle": "round,pad=0.55",
                "facecolor": facecolor,
                "edgecolor": edgecolor,
                "linewidth": 1.4,
            },
        )

    axis.set_xlim(-1.8, max(1.8, max_depth * x_spacing + 1.8))
    axis.set_ylim(-node_count - 1, 0)
    axis.axis("off")
    axis.set_title(
        f"Visualisasi Backward Chaining — Query: {query}",
        fontsize=15, fontweight="bold", pad=20,
    )
    figure.text(
        0.5, 0.005,
        "Hijau = goal berhasil dibuktikan, merah = goal gagal dibuktikan.",
        ha="center", fontsize=9, color="#4b5563",
    )
    plt.tight_layout()
    plt.show()


---
# Latihan Soal

Gunakan satu kasus yang sama untuk Soal 1-3. Tujuannya agar terlihat bahwa
forward chaining dan backward chaining dapat memakai KB Horn clause yang sama,
tetapi memulai pembuktian dari arah yang berbeda.

## Soal 1 — Membuat KB Horn Clause

Diketahui informasi berikut:

1. `A`, `B`, dan `C` adalah fact;
2. jika `A` dan `B` benar, maka `D` benar;
3. jika `B` dan `C` benar, maka `E` benar;
4. jika `D` dan `E` benar, maka `F` benar;
5. jika `F` benar, maka `G` benar.

Buat KB dengan panduan berikut:

1. tulis fact sebagai list string bernama `exercise_facts`;
2. ubah setiap kalimat "jika ... maka ..." menjadi rule dengan simbol `==>`;
3. gunakan `&` jika sebuah rule mempunyai lebih dari satu premise;
4. pastikan setiap rule mempunyai tepat satu conclusion;
5. buat `exercise_kb` dengan `make_kb(exercise_facts, exercise_rules)`;
6. tampilkan `exercise_kb.clauses` dan periksa apakah seluruh fact dan rule sudah
   masuk ke KB.

Tuliskan juga premise dan conclusion dari setiap rule yang dibuat.


In [ ]:
# Fact: sentence yang sudah diketahui benar.
exercise_facts = ["A", "B", "C"]

# Setiap "jika ... maka ..." jadi satu definite clause:
# premise digabung dengan &, conclusion tepat satu simbol positif.
exercise_rules = [
    "(A & B) ==> D",
    "(B & C) ==> E",
    "(D & E) ==> F",
    "F ==> G",
]

exercise_kb = make_kb(exercise_facts, exercise_rules)
print("Isi KB:", exercise_kb.clauses)

# Cek semua fact dan rule sudah masuk ke KB.
expected = [expr(s) for s in exercise_facts + exercise_rules]
assert all(c in exercise_kb.clauses for c in expected)
assert len(exercise_kb.clauses) == len(expected)

# Premise dan conclusion tiap rule, diambil langsung dari clause-nya.
rows = []
for rule in exercise_kb.clauses:
    if rule.op == "==>":
        premises, conclusion = parse_definite_clause(rule)
        rows.append({
            "rule": str(rule),
            "premise": ", ".join(str(x) for x in premises),
            "conclusion": str(conclusion),
        })
display(pd.DataFrame(rows))


**Jawaban Soal 1.** Isi KB ada 7 clause: 3 fact (`A`, `B`, `C`) dan 4 rule.

| Rule | Premise | Conclusion |
|---|---|---|
| `(A & B) ==> D` | `A`, `B` | `D` |
| `(B & C) ==> E` | `B`, `C` | `E` |
| `(D & E) ==> F` | `D`, `E` | `F` |
| `F ==> G` | `F` | `G` |

Semua rule adalah definite clause (Horn clause) karena masing-masing punya
tepat satu conclusion positif, jadi KB ini bisa dipakai untuk forward maupun
backward chaining.


<details>
<summary>Klik untuk melihat pembahasan</summary>

```python
exercise_facts = ["A", "B", "C"]

exercise_rules = [
    "(A & B) ==> D",
    "(B & C) ==> E",
    "(D & E) ==> F",
    "F ==> G",
]

exercise_kb = make_kb(exercise_facts, exercise_rules)
exercise_kb.clauses
```

Isi KB tersebut terdiri dari:

| Rule | Premise | Conclusion |
|---|---|---|
| `(A & B) ==> D` | `A`, `B` | `D` |
| `(B & C) ==> E` | `B`, `C` | `E` |
| `(D & E) ==> F` | `D`, `E` | `F` |
| `F ==> G` | `F` | `G` |

Semua rule merupakan definite clause karena mempunyai satu conclusion positif.

</details>


## Soal 2 — Pembuktian dengan Forward Chaining

Gunakan `exercise_kb` dari Soal 1 untuk membuktikan query `G` dengan forward
chaining. Mahasiswa cukup memanggil fungsi yang sudah dibuat setelah Setup:

```python
found_fc_exercise, trace_fc_exercise = ask_forward(exercise_kb, "G")
```

Kerjakan langkah berikut:

1. tampilkan nilai `found_fc_exercise`;
2. tampilkan tabel `trace_fc_exercise`;
3. tampilkan visualisasi keputusan dengan
   `visualize_forward(trace_fc_exercise)`;
4. tuliskan urutan fact yang diproses;
5. tuliskan rule yang menghasilkan `D`, `E`, `F`, dan `G`;
6. simpulkan apakah `KB ⊨ G`.


In [ ]:
found_fc_exercise, trace_fc_exercise = ask_forward(exercise_kb, "G")

# 1. Nilai hasil
print("G berhasil dibuktikan?", found_fc_exercise)

# 2. Tabel trace
display(trace_fc_exercise)

# 3. Visualisasi
visualize_forward(trace_fc_exercise)

# 4. Urutan fact yang diproses
urutan_fc = list(trace_fc_exercise["fact diproses"])
print("Urutan fact diproses:", " → ".join(urutan_fc))

# 5. Rule yang menghasilkan D, E, F, G
for _, row in trace_fc_exercise.iterrows():
    if row["conclusion baru"] != "-":
        print(f"  {row['rule aktif']:15} menghasilkan {row['conclusion baru']} "
              f"(aktif saat fact {row['fact diproses']} diproses)")

# 6. Kesimpulan
print("KB ⊨ G?", found_fc_exercise)


NameError: name 'ask_forward' is not defined

**Jawaban Soal 2.**

1. `found_fc_exercise` bernilai `True`.
2. dan 3. Lihat tabel dan visualisasi di atas.
4. Urutan fact yang diproses: `C → B → E → A → D → F → G`. Agenda awalnya
   `A → B → C`, dan yang diambil selalu item paling kanan, jadi `C` diproses
   duluan.
5. Rule yang menghasilkan conclusion baru:
   - `C` diproses: belum ada rule yang semua premise-nya terpenuhi.
   - `B` diproses: `(B & C) ==> E` aktif dan menghasilkan **E**.
   - `E` diproses: `(D & E) ==> F` belum aktif karena `D` belum diketahui.
   - `A` diproses: `(A & B) ==> D` aktif dan menghasilkan **D**.
   - `D` diproses: `(D & E) ==> F` aktif dan menghasilkan **F**.
   - `F` diproses: `F ==> G` aktif dan menghasilkan **G**.
   - `G` diambil dari agenda, sama dengan query, jadi proses berhenti.
6. Forward chaining berangkat dari fact, lalu terus menurunkan conclusion baru
   sampai query `G` muncul. PL-FC-ENTAILS sound, jadi $KB \vDash G$.


<details>
<summary>Klik untuk melihat pembahasan</summary>

```python
found_fc_exercise, trace_fc_exercise = ask_forward(exercise_kb, "G")

print("G berhasil dibuktikan?", found_fc_exercise)
display(trace_fc_exercise)
visualize_forward(trace_fc_exercise)
```

Hasilnya `True`. Dengan urutan agenda pada fungsi, fact yang diproses adalah:

```text
C → B → E → A → D → F → G
```

Pembuktiannya:

```text
A dan B menghasilkan D
B dan C menghasilkan E
D dan E menghasilkan F
F menghasilkan G
```

Karena query `G` berhasil diturunkan dari fact dan rule di dalam KB, maka
$KB \vDash G$.

</details>


## Soal 3 — Pembuktian dengan Backward Chaining

Gunakan `exercise_kb` dari Soal 1 dan query yang sama, yaitu `G`. Jalankan
backward chaining dengan fungsi yang sudah dibuat setelah Setup:

```python
found_bc_exercise, trace_bc_exercise = ask_backward(exercise_kb, "G")
```

Kerjakan langkah berikut:

1. tampilkan nilai `found_bc_exercise`;
2. tampilkan tabel `trace_bc_exercise`;
3. tampilkan proof-search tree dengan
   `visualize_backward(trace_bc_exercise)`;
4. tentukan root, subgoal, dan leaf fact pada tree;
5. jelaskan pembuktian dari query `G` sampai mencapai fact `A`, `B`, dan `C`;
6. simpulkan apakah `KB \vDash G`.


In [ ]:
found_bc_exercise, trace_bc_exercise = ask_backward(exercise_kb, "G")

# 1. Nilai hasil
print("G berhasil dibuktikan?", found_bc_exercise)

# 2. Tabel trace
display(trace_bc_exercise)

# 3. Proof-search tree
visualize_backward(trace_bc_exercise)

# 4. Root, subgoal, dan leaf fact, dibaca dari trace
root = trace_bc_exercise.loc[trace_bc_exercise["depth"] == 0, "goal"].iloc[0]
is_leaf = trace_bc_exercise["keputusan"] == "Goal sudah tersedia sebagai fact."
subgoal = trace_bc_exercise.loc[(trace_bc_exercise["depth"] > 0) & ~is_leaf, "goal"]
leaf = trace_bc_exercise.loc[is_leaf, "goal"]
print("Root      :", root)
print("Subgoal   :", ", ".join(dict.fromkeys(subgoal)))
print("Leaf fact :", ", ".join(dict.fromkeys(leaf)))

# 5. Jalur pembuktian dari G sampai fact
for _, row in trace_bc_exercise.iterrows():
    indent = "    " * row["depth"]
    via = f" lewat {row['rule berhasil']}" if row["rule berhasil"] != "-" else ""
    print(f"{indent}{row['goal']}: {row['keputusan']}{via}")

# 6. Kesimpulan
print("KB ⊨ G?", found_bc_exercise)


**Jawaban Soal 3.**

1. `found_bc_exercise` bernilai `True`.
2. dan 3. Lihat tabel dan proof-search tree di atas.
4. Bagian tree:
   - **root**: `G` (query);
   - **subgoal**: `F`, `D`, `E`;
   - **leaf fact**: `A`, `B`, `C`. `B` muncul dua kali, sebagai premise `D`
     dan sebagai premise `E`.
5. Pembuktian dari query ke fact:

```text
G  ← F ==> G           butuh F
└── F  ← (D & E) ==> F  butuh D dan E
    ├── D  ← (A & B) ==> D
    │   ├── A  fact ✓
    │   └── B  fact ✓
    └── E  ← (B & C) ==> E
        ├── B  fact ✓
        └── C  fact ✓
```

   Backward chaining mulai dari `G` lalu mencari rule yang conclusion-nya `G`,
   yaitu `F ==> G`, sehingga `F` jadi subgoal. `F` dihasilkan
   `(D & E) ==> F`, jadi `D` dan `E` jadi subgoal. `D` butuh `A` dan `B`,
   `E` butuh `B` dan `C`, dan keempatnya sudah ada di KB sebagai fact. Setelah
   semua leaf terbukti, hasilnya dinaikkan lagi: `D` dan `E` benar, jadi `F`
   benar, jadi `G` benar.
6. Semua subgoal berujung di fact, jadi $KB \vDash G$. Hasilnya sama dengan
   forward chaining di Soal 2. Bedanya cuma arah: FC mulai dari fact dan
   memproses semua yang bisa diturunkan, sedangkan BC mulai dari query dan hanya
   memeriksa clause yang relevan dengan `G`.


<details>
<summary>Klik untuk melihat pembahasan</summary>

```python
found_bc_exercise, trace_bc_exercise = ask_backward(exercise_kb, "G")

print("G berhasil dibuktikan?", found_bc_exercise)
display(trace_bc_exercise)
visualize_backward(trace_bc_exercise)
```

Hasilnya `True`. Backward chaining memulai pencarian dari query `G`:

```text
G
└── membutuhkan F
    ├── membutuhkan D
    │   ├── A adalah fact
    │   └── B adalah fact
    └── membutuhkan E
        ├── B adalah fact
        └── C adalah fact
```

- **root** pada tree adalah `G`;
- **subgoal** adalah `F`, `D`, dan `E`;
- **leaf fact** adalah `A`, `B`, dan `C`.

Karena seluruh leaf yang diperlukan tersedia sebagai fact, `D` dan `E`
berhasil dibuktikan. Keduanya membuktikan `F`, lalu `F` membuktikan `G`.
Dengan demikian, $KB \vDash G$.

</details>
